In [1]:
#!pip uninstall -y gradio gradio_client huggingface_hub transformers sentence-transformers

In [2]:
#!pip install -U huggingface_hub
#!pip install -U transformers
#!pip install -U sentence-transformers

#!pip install gradio==5.32.1

#!pip install pandas html2text

In [3]:
#!pip install xxhash

In [4]:
import gradio as gr
import html
import uuid
import json
import pandas as pd
from collections import defaultdict

In [5]:
import re                               # for document class
import xml.etree.ElementTree as ET      # for parsing xml
import os
import pandas as pd
# text processor
#import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
# end text processor
import shutil                             # copy dataset
# for dbscan
from sklearn.cluster import DBSCAN
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
import time
from collections import defaultdict
# for ngram hashing
import hashlib
# semantic analyzer
import torch
from sentence_transformers import SentenceTransformer
# multi farme vizualization
import xml.etree.ElementTree as ET
from xml.dom import minidom
import random
import xxhash

In [6]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive', force_remount=True)

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

print(f"Lookin for data in: {BASE_PATH}")

Mounted at /content/drive
Lookin for data in: /content/drive/MyDrive/PAN11/external-detection-corpus


In [72]:
SBERT_THRESHOLD = 0.65
EPS = 2000
MIN_SAMPLE = 2
CONFIDENCE_THRESHOLD = 0.75
MAX_OCC = 15

MAX_ANCHORS = 1000
FRAGMENT_WINDOW = 300
BATCH_SIZE = 1024
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'

In [53]:
%%writefile word_encoplot.cpp

#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>
#include <cctype>
#include <stdint.h>

struct Token {
    uint64_t hash;
    int offset;
};

struct NGram {
    uint64_t hash;
    int offset;
};

static uint64_t fnv_word(const std::string& s) {
    uint64_t h = 1469598103934665603ULL;
    for (unsigned char c : s) {
        h ^= c;
        h *= 1099511628211ULL;
    }
    return h;
}

static uint64_t fnv_combine(uint64_t h, uint64_t x) {
    h ^= x;
    h *= 1099511628211ULL;
    return h;
}

std::vector<Token> tokenize(const std::string& text) {
    std::vector<Token> result;
    std::string word;
    int start = -1;

    for (int i = 0; i < (int)text.size(); i++) {
        char c = text[i];

        if (std::isalnum((unsigned char)c) || c == '_') {
            if (word.empty())
                start = i;
            word.push_back(std::tolower(c));
        }
        else {
            if (!word.empty()) {
                result.push_back({
                    fnv_word(word),
                    start
                });
                word.clear();
            }
        }
    }

    if (!word.empty()) {
        result.push_back({
            fnv_word(word),
            start
        });
    }

    return result;
}

std::vector<NGram> build_ngrams(const std::vector<Token>& tokens, int n) {
    std::vector<NGram> result;
    if ((int)tokens.size() < n)
        return result;

    for (int i = 0; i <= (int)tokens.size() - n; i++) {
        uint64_t h = 1469598103934665603ULL;
        for (int j = 0; j < n; j++) {
            h = fnv_combine(h, tokens[i+j].hash);
        }
        result.push_back({
            h,
            tokens[i].offset
        });
    }
    return result;
}

std::string read_file(const char* path) {
    std::ifstream f(path);
    if (!f.good())
        return "";

    return std::string(
        std::istreambuf_iterator<char>(f),
        std::istreambuf_iterator<char>()
    );
}

int main(int argc, char** argv) {
    if (argc != 3)
        return 1;

    std::string susp_text = read_file(argv[1]);
    std::string src_text  = read_file(argv[2]);

    auto susp_tokens = tokenize(susp_text);
    auto src_tokens  = tokenize(src_text);

    auto susp_ngrams = build_ngrams(susp_tokens, 4);
    auto src_ngrams  = build_ngrams(src_tokens, 4);

    std::sort(susp_ngrams.begin(), susp_ngrams.end(), [](const NGram& a, const NGram& b) {
        return a.hash < b.hash;
    });

    std::sort(src_ngrams.begin(), src_ngrams.end(), [](const NGram& a, const NGram& b) {
        return a.hash < b.hash;
    });

    const int MAX_OCC = 15;

    size_t i = 0;
    size_t j = 0;

    while (i < susp_ngrams.size() && j < src_ngrams.size()) {
        uint64_t h1 = susp_ngrams[i].hash;
        uint64_t h2 = src_ngrams[j].hash;

        if (h1 == h2) {
            size_t i_start = i;
            size_t j_start = j;

            while (i < susp_ngrams.size() && susp_ngrams[i].hash == h1) {
                i++;
            }

            while (j < src_ngrams.size() && src_ngrams[j].hash == h2) {
                j++;
            }

            size_t susp_count = i - i_start;
            size_t src_count  = j - j_start;

            if (susp_count <= MAX_OCC && src_count <= MAX_OCC) {
                for (size_t a = i_start; a < i; a++) {
                    for (size_t b = j_start; b < j; b++) {
                        std::cout << susp_ngrams[a].offset << " " << src_ngrams[b].offset << "\n";
                    }
                }
            }
        }
        else if (h1 < h2) {
            while (i < susp_ngrams.size() && susp_ngrams[i].hash == h1) {
                i++;
            }
        }
        else {
            while (j < src_ngrams.size() && src_ngrams[j].hash == h2) {
                j++;
            }
        }
    }

    return 0;
}

Overwriting word_encoplot.cpp


In [54]:
!g++ -O3 -march=native word_encoplot.cpp -o word_encoplot

In [10]:
class Document:
  def __init__(self, doc_id, is_source=True, text_content=None):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    if text_content is not None:
      self.text = text_content
      self.doc_name = f"demo_doc_{doc_id}.txt"
      self.xml_path = None
      self.file_path = None
      self.metadata = {}
      self.language = 'english'
    else:
      formatted_id = f"{self.numeric_id:05d}"
      prefix = "source-document" if is_source else "suspicious-document"

      part_number = ((self.numeric_id - 1) // 500) + 1
      part_folder = f"part{part_number}"
      base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

      self.doc_name = f"{prefix}{formatted_id}.txt"
      self.xml_name = f"{prefix}{formatted_id}.xml"

      self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
      self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

      with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
        self.text = f.read()

      self.metadata = self._parse_xml()
      self.language = self.metadata.get('lang', 'english')

    self.segments = []

  def _parse_xml(self):
    if self.xml_path is None:
        return {}
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

In [11]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

In [12]:
class PredictedSegment:
    def __init__(self, susp_id, src_id, susp_off, susp_len, src_off, src_len, score):
        self.susp_id = susp_id
        self.src_id = src_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_off = src_off
        self.src_len = src_len
        self.score = score

    def __repr__(self):
        return f"<Match Susp:{self.susp_id} Src:{self.src_id} Score:{self.score:.2f}>"

In [57]:
import subprocess

class WordEncoplotEngine:

    def __init__(
        self,
        executable="./word_encoplot",
        max_anchors=MAX_ANCHORS
    ):
        self.executable = executable
        self.max_anchors = max_anchors

    def compute_document_hashes(
        self,
        document
    ):
        return None

    def get_anchors_for_pair(
        self,
        susp_doc,
        src_doc,
        susp_hashes=None,
        src_hashes=None
    ):

        result = subprocess.run(
            [
                self.executable,
                susp_doc.file_path,
                src_doc.file_path
            ],
            capture_output=True,
            text=True
        )

        anchors = []

        for line in result.stdout.splitlines():

            parts = line.strip().split()

            if len(parts) != 2:
                continue

            anchors.append([
                int(parts[0]),
                int(parts[1])
            ])

        anchors = self.apply_bucketing(
            anchors,
            len(susp_doc.text)
        )

        anchors.sort(
            key=lambda x: x[0]
        )

        return anchors

    def run_mass_scan(self, susp_path, source_paths):
        anchors_by_source = {path: [] for path in source_paths}

        for src_path in source_paths:
            result = subprocess.run(
                [self.executable, susp_path, src_path],
                capture_output=True,
                text=True
            )

            anchors = []
            for line in result.stdout.splitlines():
                parts = line.strip().split()
                if len(parts) != 2:
                    continue
                anchors.append([int(parts[0]), int(parts[1])])

            anchors.sort(key=lambda x: x[0])
            anchors_by_source[src_path] = anchors

        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        if len(anchors) <= self.max_anchors:
            return anchors

        bucket_size = max(
            1,
            doc_len // self.max_anchors
        )

        buckets = {}

        for p_susp, p_src in anchors:

            idx = p_susp // bucket_size

            if idx not in buckets:
                buckets[idx] = (
                    p_susp,
                    p_src
                )

        return list(
            buckets.values()
        )

In [58]:
engine = WordEncoplotEngine()

In [40]:
class NgramRadarEngine:
    def __init__(self, n_gram_size=3, max_anchors=MAX_ANCHORS):
        self.n = n_gram_size
        self.max_anchors = max_anchors

    def _get_ngrams_with_offsets(self, text):
        words_iter = re.finditer(r'\b\w+\b', text.lower())
        words = [(m.group(0), m.start()) for m in words_iter]

        ngram_map = defaultdict(list)

        if len(words) < self.n:
            return ngram_map

        for i in range(len(words) - self.n + 1):
            ngram_phrase = " ".join([w[0] for w in words[i:i+self.n]])
            ngram_hash = hashlib.md5(ngram_phrase.encode('utf-8')).hexdigest()[:12]
            start_offset = words[i][1]
            ngram_map[ngram_hash].append(start_offset)

        return ngram_map

    def compute_document_hashes(self, document):
        return self._get_ngrams_with_offsets(document.text)

    def get_anchors_for_pair(self, susp_doc, src_doc, susp_ngrams, src_ngrams):
        anchors = []

        common_hashes = set(susp_ngrams.keys()).intersection(set(src_ngrams.keys()))

        for h in common_hashes:
            susp_occ = susp_ngrams[h]
            src_occ = src_ngrams[h]

            if len(susp_occ) <= MAX_OCC and len(src_occ) <= MAX_OCC:
                for s_off in susp_occ:
                    for src_off in src_occ:
                        anchors.append([s_off, src_off])

        # Apply bucketing
        anchors = self.apply_bucketing(anchors, len(susp_doc.text))

        # Sort anchors by suspicious offset
        anchors.sort(key=lambda x: x[0])
        return anchors

    def run_mass_scan(self, susp_path, source_paths):
        anchors_by_source = {path: [] for path in source_paths}

        try:
            with open(susp_path, "r", encoding='utf-8', errors='ignore') as f:
                susp_text = f.read()
        except Exception:
            return anchors_by_source

        susp_ngrams = self._get_ngrams_with_offsets(susp_text)

        if not susp_ngrams:
            return anchors_by_source

        for src_path in source_paths:
            try:
                with open(src_path, "r", encoding='utf-8', errors='ignore') as f:
                    src_text = f.read()
            except Exception:
                continue

            src_ngrams = self._get_ngrams_with_offsets(src_text)

            common_hashes = set(susp_ngrams.keys()).intersection(set(src_ngrams.keys()))

            for h in common_hashes:
                susp_occ = susp_ngrams[h]
                src_occ = src_ngrams[h]

                if len(susp_occ) <= MAX_OCC and len(src_occ) <= MAX_OCC:
                    for s_off in susp_occ:
                        for src_off in src_occ:
                            anchors_by_source[src_path].append([s_off, src_off])

            anchors_by_source[src_path].sort(key=lambda x: x[0])

        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        if len(anchors) <= self.max_anchors:
            return anchors

        bucket_size = max(1, doc_len // self.max_anchors)
        buckets = {}
        for p_susp, p_src in anchors:
            bucket_idx = p_susp // bucket_size
            if bucket_idx not in buckets:
                buckets[bucket_idx] = (p_susp, p_src)
        return list(buckets.values())


In [16]:
class SemanticAnalyzer:
  def __init__(self, model_name=MODEL_NAME, progress=None):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    if progress is not None:
        progress(0.05, desc=f"Loading SBERT model ({model_name}) into memory...")

    print(f"[INIT] Loading {model_name}...")
    self.model = SentenceTransformer(model_name)

    if self.device == "cuda":
        print("[INIT] Converting model to FP16 (Mixed Precision) for T4 Tensor Cores...")
        self.model = self.model.half()

    self.model = self.model.to(self.device)

  def verify_anchors(self, susp_doc, src_doc, anchors, progress=None):
    # Forward to the new multiple-source method for consistency
    return self.verify_multiple(susp_doc, [(src_doc, anchors)], progress)

  def verify_multiple(self, susp_doc, source_anchor_pairs, progress=None):
    all_texts_susp = []
    all_texts_src = []
    all_meta = []

    for src_doc, anchors in source_anchor_pairs:
        for s_off, src_off in anchors:
            f_susp, s_off_actual, s_len_actual = get_safe_fragment(susp_doc.text, s_off, FRAGMENT_WINDOW)
            f_src, src_off_actual, src_len_actual = get_safe_fragment(src_doc.text, src_off, FRAGMENT_WINDOW)

            if not f_susp or not f_src:
                continue

            t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
            t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

            if len(t_susp) > 30:
                all_texts_susp.append(t_susp)
                all_texts_src.append(t_src)
                all_meta.append({
                    'susp_id': susp_doc.numeric_id,
                    'src_id': src_doc.numeric_id,
                    's_off': s_off_actual, 's_len': s_len_actual,
                    'src_off': src_off_actual, 'src_len': src_len_actual,
                })

    if not all_texts_susp:
        return [], [], np.array([])

    if progress is not None:
        emb_susp_list = []
        emb_src_list = []
        num_chunks = math.ceil(len(all_texts_susp) / BATCH_SIZE)

        for i in progress.tqdm(range(num_chunks), desc=f"SBERT Encoding {len(all_texts_susp)} Pairs"):
            start_idx = i * BATCH_SIZE
            end_idx = min((i + 1) * BATCH_SIZE, len(all_texts_susp))

            chunk_susp = all_texts_susp[start_idx:end_idx]
            chunk_src = all_texts_src[start_idx:end_idx]

            e_s = self.model.encode(chunk_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)
            e_r = self.model.encode(chunk_src, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)

            emb_susp_list.append(e_s)
            emb_src_list.append(e_r)

        emb_susp = np.vstack(emb_susp_list)
        emb_src = np.vstack(emb_src_list)
    else:
        emb_susp = self.model.encode(all_texts_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)
        emb_src  = self.model.encode(all_texts_src,  convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)

    scores = np.zeros(len(all_texts_susp), dtype=np.float32)
    chunk_size = 50000
    for i in range(0, len(all_texts_susp), chunk_size):
        j = min(i + chunk_size, len(all_texts_susp))
        dot = np.sum(emb_susp[i:j] * emb_src[i:j], axis=1)
        n_s = np.linalg.norm(emb_susp[i:j], axis=1)
        n_src = np.linalg.norm(emb_src[i:j], axis=1)
        scores[i:j] = dot / (n_s * n_src + 1e-8)

    del emb_susp
    del emb_src
    torch.cuda.empty_cache()

    verified_fragments = []
    for i, score in enumerate(scores):
        if score >= SBERT_THRESHOLD:
            m = all_meta[i]
            verified_fragments.append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score)
            )
    return verified_fragments, all_meta, scores

In [17]:
class PlagiarismCase:
    def __init__(self, susp_id, susp_off, susp_len, src_id, src_off, src_len, score):
        self.susp_id  = susp_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_id   = src_id
        self.src_off  = src_off
        self.src_len  = src_len
        self.score    = score

    def __repr__(self):
        return (f"PlagiarismCase(susp={self.susp_id}, src={self.src_id}, "
                f"susp_off={self.susp_off}, src_off={self.src_off}, "
                f"score={self.score:.2f})")

In [18]:
class ValidationMetrics:
    def __init__(self):
        pass

    def compute_granularity(self, gt_fragments, detected_fragments):
        if not gt_fragments:
            return 1.0
        total_gran = 0
        for gt_start, gt_end in gt_fragments:
            overlapping = 0
            for det_start, det_end in detected_fragments:
                if det_start < gt_end and det_end > gt_start:
                    overlapping += 1
            total_gran += max(1, overlapping)
        return total_gran / len(gt_fragments)

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        metrics = {
            "recall": 0.0,
            "precision": 0.0,
            "f1": 0.0,
            "granularity": 1.0,
            "plagdet": 0.0
        }

        if not ground_truth_features:
            metrics["recall"] = 1.0
            return metrics

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)
            total_recall += (covered_len / gt.this_length)
        metrics["recall"] = total_recall / len(ground_truth_features)

        # precision
        if predicted_segments:
            total_precision = 0
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            metrics["precision"] = total_precision / len(predicted_segments)

        # granularity
        gt_tuples = [(f.this_offset, f.this_offset + f.this_length) for f in ground_truth_features]
        det_tuples = [(d.susp_off, d.susp_off + d.susp_len) for d in predicted_segments]
        metrics["granularity"] = self.compute_granularity(gt_tuples, det_tuples)

        # f1 and plagDet
        if metrics["precision"] + metrics["recall"] > 0:
            metrics["f1"] = 2 * (metrics["precision"] * metrics["recall"]) / (metrics["precision"] + metrics["recall"])

        metrics["plagdet"] = metrics["f1"] / math.log2(1 + metrics["granularity"]) if metrics["granularity"] > 0 else 0

        return {k: round(v, 4) for k, v in metrics.items()}


    def calculate_global_score(self, all_results_list):
      relevant_results = [res for res in all_results_list if not (res['recall'] == 1.0 and res['precision'] == 0.0)]

      if not relevant_results:
        print("\n" + "!"*60)
        print("  NO RELEVANT DATA TO CALCULATE GLOBAL SCORE")
        print("  (All documents were original and correctly identified as such)")
        print("!"*60)
        return None

      n = len(relevant_results)
      avg_metrics = {k: sum(res[k] for res in relevant_results) / n for k in relevant_results[0].keys()}

      print("\n" + "="*60)
      print(f"GLOBAL SCORE (average for {n} relevant documents)")
      print("-" * 60)
      print(f"  Precision    : {avg_metrics['precision']:.4f}")
      print(f"  Recall       : {avg_metrics['recall']:.4f}")
      print(f"  F1-Score     : {avg_metrics['f1']:.4f}")
      print(f"  Granularity  : {avg_metrics['granularity']:.4f}")
      print(f"  PlagDet      : {avg_metrics['plagdet']:.4f}")
      print("="*60 + "\n")

      return avg_metrics

    def analyze_by_obfuscation(self, suspicious_docs, detections):
        import math

        obf_data = {}

        for doc in suspicious_docs:
            susp_id = doc.numeric_id
            my_dets = detections.get(susp_id, [])

            for gt in doc.plagiarism_features:
                obf_type = gt.obfuscation if gt.obfuscation else "unknown"

                if obf_type not in obf_data:
                    obf_data[obf_type] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[obf_type]['gt_count'] += 1
                gt_start, gt_end = gt.this_offset, gt.this_offset + gt.this_length
                obf_data[obf_type]['gt_fragments'].append((gt_start, gt_end))

                for c in range(gt_start, gt_end):
                    obf_data[obf_type]['gt_chars'].add((susp_id, c))

            for det in my_dets:
                det_start, det_end = det.susp_off, det.susp_off + det.susp_len
                assigned_obf = None
                max_overlap = 0

                for gt in doc.plagiarism_features:
                    overlap = max(0, min(gt.this_offset + gt.this_length, det_end) - max(gt.this_offset, det_start))
                    if overlap > max_overlap:
                        max_overlap = overlap
                        assigned_obf = gt.obfuscation if gt.obfuscation else "unknown"

                if assigned_obf is None:
                    assigned_obf = "false_positive_only"

                if assigned_obf not in obf_data:
                    obf_data[assigned_obf] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[assigned_obf]['det_count'] += 1
                obf_data[assigned_obf]['det_fragments'].append((det_start, det_end))
                for c in range(det_start, det_end):
                    obf_data[assigned_obf]['det_chars'].add((susp_id, c))

        print("\n" + "="*85)
        print("DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE")
        print("-" * 85)
        print(f"{'Obfuscation Type':<18} | {'GT/Det':<8} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Gran':<6} | {'PlagDet':<8}")
        print("-" * 85)

        for obf_type, data in obf_data.items():
            if obf_type == "false_positive_only":
                continue

            gt_chars = data['gt_chars']
            det_chars = data['det_chars']

            tp_chars = len(gt_chars & det_chars)

            precision = tp_chars / len(det_chars) if len(det_chars) > 0 else 0.0
            recall = tp_chars / len(gt_chars) if len(gt_chars) > 0 else 0.0

            f1 = 0.0
            if precision + recall > 0:
                f1 = 2 * (precision * recall) / (precision + recall)

            gran = self.compute_granularity(data['gt_fragments'], data['det_fragments'])

            plagdet = f1 / math.log2(1 + gran) if gran > 0 else 0.0

            print(f"{obf_type:<18} | {data['gt_count']}/{data['det_count']:<6} | {precision:.4f}    | {recall:.4f} | {f1:.4f}   | {gran:.2f} | {plagdet:.4f}")

        print("="*85 + "\n")


In [19]:
def get_safe_fragment(text, offset, win=FRAGMENT_WINDOW):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < len(text) and text[end] not in [' ', '\n', '.', '!', '?']:
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

In [20]:
def extract_sample(n_suspicious):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(1, n_suspicious + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(source_ids_needed)

In [21]:
def extract_sources_for_suspicious(susp_id, n_sources, source_pool_size=11093):
    doc = Document(susp_id, is_source=False)

    required_ids = set()
    for pf in doc.plagiarism_features:
        src_id = int(pf.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))
        required_ids.add(src_id)

    print(f"[SOURCES] Doc {susp_id:05d} | "
          f"GT sources: {len(required_ids)} | "
          f"Target total: {n_sources}")

    if len(required_ids) >= n_sources:
        print(f"[SOURCES] GT sources ({len(required_ids)}) >= n_sources ({n_sources}), returning all GT sources.")
        return doc, sorted(required_ids)

    all_possible = set(range(1, source_pool_size + 1)) - required_ids
    n_random     = n_sources - len(required_ids)
    random_ids   = set(random.sample(sorted(all_possible), n_random))

    final_ids = sorted(required_ids | random_ids)

    print(f"[SOURCES] GT: {sorted(required_ids)} | "
          f"Random fill: {n_random} | "
          f"Total: {len(final_ids)}")

    return doc, final_ids

In [77]:
def cluster_results_dbscan(hits, eps=EPS, min_samples=MIN_SAMPLE):
    if not hits:
        return []

    hits_by_source = defaultdict(list)
    for h in hits:
        hits_by_source[h.src_id].append(h)

    results = []

    for src_id, src_hits in hits_by_source.items():

        points = np.array([[h.susp_off, h.src_off] for h in src_hits])
        db     = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
        labels = db.labels_

        for label in set(labels):
            if label == -1:
                continue

            cluster = [src_hits[i] for i, l in enumerate(labels) if l == label]

            susp_min  = min(h.susp_off for h in cluster)
            susp_max  = max(h.susp_off + h.susp_len for h in cluster)
            src_min   = min(h.src_off for h in cluster)
            src_max   = max(h.src_off + h.src_len for h in cluster)
            avg_score = sum(h.score for h in cluster) / len(cluster)

            results.append(PlagiarismCase(
                susp_id  = cluster[0].susp_id,
                susp_off = susp_min,
                susp_len = susp_max - susp_min,
                src_id   = src_id,
                src_off  = src_min,
                src_len  = src_max - src_min,
                score    = avg_score
            ))

    return results

In [24]:
def debug_plagiarism(doc_susp, detections, all_meta, scores_sbert=None, threshold=SBERT_THRESHOLD):
    print(f"DETAILED ANALYSIS: {doc_susp.doc_name} ---")

    my_detections = detections.get(doc_susp.numeric_id, [])

    if doc_susp.plagiarism_features:
        print(f"Ground Truth properties: {len(doc_susp.plagiarism_features)} plagiarized fragments.")
        for pf in doc_susp.plagiarism_features:
            gt_start = pf.this_offset
            gt_end = pf.this_offset + pf.this_length
            src_target = pf.get_source_id()

            ngram_hits = [m for m in all_meta if m['susp_id'] == doc_susp.numeric_id
                          and m['src_id'] == src_target
                          and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]

            best_score = 0.0
            if scores_sbert is not None:
                relevant_indices = [i for i, m in enumerate(all_meta)
                                   if m['susp_id'] == doc_susp.numeric_id
                                   and m['src_id'] == src_target
                                   and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
                if relevant_indices:
                    best_score = max([scores_sbert[i].item() for i in relevant_indices])

            final_det = [d for d in my_detections if d.src_id == src_target
                         and (d.susp_off >= gt_start - 500 and d.susp_off <= gt_end + 500)]

            status = " DETECTED" if final_det else " MISSED"
            print(f"\nGT Fragment [{gt_start}:{gt_end}] (Source {src_target:05d}) -> {status}")
            print(f"   - Anchors generated (N-gram): {len(ngram_hits)}")

            if len(ngram_hits) > 0:
                print(f"   - Best SBERT score:           {best_score:.4f}")

            if not final_det:
                if not ngram_hits:
                    print("   [CAUSE]: Lexical Failure - N-gram Engine found no anchors.")
                else:
                    print(f"   [CAUSE]: Semantic Failure - Score {best_score:.4f} < {threshold} or DBSCAN removed it.")
    else:
        print("Original document (according to Ground Truth).")

    for det in my_detections:
        is_real = False
        for pf in doc_susp.plagiarism_features:
            if not (det.susp_off + det.susp_len < pf.this_offset or pf.this_offset + pf.this_length < det.susp_off):
                is_real = True
                break

        if not is_real:
            print(f"\n FALSE POSITIVE DETECTED (Wrongly reported as plagiarism):")
            print(f"   - Position: [{det.susp_off}:{det.susp_off + det.susp_len}] | Source: {det.src_id:05d}")
            print(f"   - SBERT score of the error: {det.score:.4f}")

In [25]:
analyzer = SemanticAnalyzer()
validator = ValidationMetrics()

[INIT] Loading paraphrase-multilingual-mpnet-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[INIT] Converting model to FP16 (Mixed Precision) for T4 Tensor Cores...


In [26]:
#engine = NgramRadarEngine(n_gram_size=4, max_anchors=MAX_ANCHORS)

In [59]:
def run_pipeline(n_suspicious, analyzer=None):
    if analyzer is None:
        if 'analyzer' in globals():
            analyzer = globals()['analyzer']
            print("[INFO] Using existing analyzer found in memory.")
        else:
            print("[INFO] Initializing new SemanticAnalyzer...")
            analyzer = SemanticAnalyzer()

    print("-" * 80)
    print(f"[STEP 1] Extracting sample: {n_suspicious} suspicious files")
    print("-" * 80)
    suspicious_docs, source_ids = extract_sample(n_suspicious)
    print(f"Necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs...")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error loading source {s_id}: {e}")
    print(f"Sources loaded: {len(source_docs)}\n")

    print("[STEP 3] N-gram Hashing - Native Python Lexical Radar")
    t3 = time.time()

    all_texts_susp = []
    all_texts_src = []
    all_meta = []

    for doc_susp in suspicious_docs:
        src_paths = [d.file_path for d in source_docs]
        raw_results = engine.run_mass_scan(doc_susp.file_path, src_paths)

        for s_path, anchors in raw_results.items():
            if not anchors: continue

            raw_anchor_count = len(anchors)
            anchors = engine.apply_bucketing(anchors, len(doc_susp.text))
            s_id = int(re.search(r'document(\d+)', s_path).group(1))

            for p_susp, p_src in anchors:
                f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
                doc_src = next((d for d in source_docs if d.numeric_id == s_id), None)
                if not doc_src: continue
                f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                if not f_susp or not f_src: continue

                t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                if len(t_susp) > 30:
                    all_texts_susp.append(t_susp)
                    all_texts_src.append(t_src)
                    all_meta.append({
                        'susp_id': doc_susp.numeric_id, 'src_id': s_id,
                        's_off': s_off, 's_len': s_len,
                        'src_off': src_off, 'src_len': src_len,
                        'raw_anchors': raw_anchor_count
                    })

    t3_elapsed = time.time() - t3
    print(f"Candidate fragments from N-gram Engine: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    print(f"[STEP 4] SBERT batch encoding for {len(all_texts_susp)} pairs...")
    t5 = time.time()

    emb_susp = analyzer.model.encode(all_texts_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)
    emb_src  = analyzer.model.encode(all_texts_src,  convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)

    print("[INFO] Computing cosine similarities in chunks")
    scores = np.zeros(len(all_texts_susp), dtype=np.float32)
    chunk_size = 50000
    for i in range(0, len(all_texts_susp), chunk_size):
        j = min(i + chunk_size, len(all_texts_susp))
        dot = np.sum(emb_susp[i:j] * emb_src[i:j], axis=1)
        n_s = np.linalg.norm(emb_susp[i:j], axis=1)
        n_src = np.linalg.norm(emb_src[i:j], axis=1)
        scores[i:j] = dot / (n_s * n_src + 1e-8)

    del emb_susp
    del emb_src
    torch.cuda.empty_cache()

    print(f"Encoding done in {time.time() - t5:.1f}s\n")

    print("[STEP 5] Filtering and clustering with DBSCAN")
    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= SBERT_THRESHOLD:
            m = all_meta[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score)
            )

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results_dbscan(hits)
        detections[susp_id].extend(clustered)
    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[STEP 6] Generating XML output...")
    os.makedirs("/content/output", exist_ok=True)
    for doc_susp in suspicious_docs:
        root = ET.Element("document", reference=doc_susp.doc_name)
        for det in detections.get(doc_susp.numeric_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det.susp_off), this_length=str(det.susp_len),
                source_reference=f"source-document{det.src_id:05d}.txt",
                source_offset=str(det.src_off), source_length=str(det.src_len)
            )
        ET.ElementTree(root).write(f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}", encoding='utf-8', xml_declaration=True)

    all_results = []
    print("[STEP 7] Detected vs ground truth...")
    print("-" * 60)

    for doc_susp in suspicious_docs:
        m = validator.calculate_metrics(doc_susp.plagiarism_features, detections.get(doc_susp.numeric_id, []))
        all_results.append(m)
        if len(doc_susp.plagiarism_features) > 0 or len(detections.get(doc_susp.numeric_id, [])) > 0:
            print(f"  Suspicious {doc_susp.numeric_id:05d} | GT: {len(doc_susp.plagiarism_features)} | "
                  f"Det: {len(detections.get(doc_susp.numeric_id, []))} | P: {m['precision']:.2f} R: {m['recall']:.2f}")

    validator.calculate_global_score(all_results)

    print("\n[STEP 9] Running obfuscation analysis...")
    validator.analyze_by_obfuscation(suspicious_docs, detections)

    print("\n" + "="*40)
    print("DETAILED TRACING OF GROUND TRUTH SEGMENTS")
    print("="*40)
    for doc_susp in suspicious_docs:
        debug_plagiarism(doc_susp, detections, all_meta, scores, SBERT_THRESHOLD)

    return suspicious_docs, source_docs, detections, scores, all_meta

In [63]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=5,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 5 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 5
Unique sources : 1
Necessary sources: 1

[STEP 2] Loading source docs...
Sources loaded: 1

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from N-gram Engine: 140 | Time: 0.0min

[STEP 4] SBERT batch encoding for 140 pairs...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 0.5s

[STEP 5] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 1

[STEP 6] Generating XML output...
[STEP 7] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99

GLOBAL SCORE (average for 1 relevant documents)
------------------------------------------------------------
  Precision    : 0.9159
  Recall       : 0.9929
  F1-Score     : 0.9529
  Granularity  : 1.0000
  PlagDet      : 0.9529


[STEP 9] Running obfuscation analysis...

DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE
-------------------------------------------------------------------------------------
Obfuscation Type   | GT/Det   | Precision  | Recall     | F1-Score   | Gran   | PlagDet 
-------------------------------------------------------------------------------------
low                | 1/1      | 0.9159    | 0.9929 | 0.9529   | 1.00 | 

In [74]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=10,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 10 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 10
Unique sources : 3
Necessary sources: 3

[STEP 2] Loading source docs...
Sources loaded: 3

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from N-gram Engine: 5712 | Time: 0.0min

[STEP 4] SBERT batch encoding for 5712 pairs...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 19.2s

[STEP 5] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 6

[STEP 6] Generating XML output...
[STEP 7] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99
  Suspicious 00007 | GT: 6 | Det: 4 | P: 0.99 R: 0.49
  Suspicious 00010 | GT: 6 | Det: 6 | P: 1.00 R: 0.29

GLOBAL SCORE (average for 3 relevant documents)
------------------------------------------------------------
  Precision    : 0.9678
  Recall       : 0.5899
  F1-Score     : 0.6844
  Granularity  : 1.1667
  PlagDet      : 0.6484


[STEP 9] Running obfuscation analysis...

DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE
-------------------------------------------------------------------------------------
Obfuscation Type   | GT/Det   | Precision  | Recall     | F1-Score   | Gran   | PlagDet 
------------------------------------------------

In [61]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=30,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 30 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 30
Unique sources : 21
Necessary sources: 21

[STEP 2] Loading source docs...
Sources loaded: 21

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from N-gram Engine: 50278 | Time: 0.3min

[STEP 4] SBERT batch encoding for 50278 pairs...


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 159.0s

[STEP 5] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 53

[STEP 6] Generating XML output...
[STEP 7] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00004 | GT: 0 | Det: 2 | P: 0.00 R: 1.00
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99
  Suspicious 00007 | GT: 6 | Det: 4 | P: 0.99 R: 0.49
  Suspicious 00010 | GT: 6 | Det: 7 | P: 1.00 R: 0.29
  Suspicious 00011 | GT: 12 | Det: 19 | P: 0.96 R: 0.66
  Suspicious 00012 | GT: 14 | Det: 23 | P: 0.99 R: 0.55
  Suspicious 00014 | GT: 3 | Det: 3 | P: 0.63 R: 0.89
  Suspicious 00015 | GT: 11 | Det: 5 | P: 0.87 R: 0.41
  Suspicious 00016 | GT: 1 | Det: 1 | P: 0.96 R: 0.97
  Suspicious 00020 | GT: 3 | Det: 3 | P: 1.00 R: 0.44
  Suspicious 00024 | GT: 3 | Det: 4 | P: 0.46 R: 0.64
  Suspicious 00025 | GT: 1 | Det: 1 | P: 1.00 R: 0.15
  Suspicious 00026 | GT: 2 | Det: 0 | P: 0.00 R: 0.00
  Suspi

In [62]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections, cache_scores, cache_meta = run_pipeline(
    n_suspicious=50,
    analyzer=analyzer
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 50 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 50
Unique sources : 51
Necessary sources: 51

[STEP 2] Loading source docs...
Sources loaded: 51

[STEP 3] N-gram Hashing - Native Python Lexical Radar
Candidate fragments from N-gram Engine: 199147 | Time: 1.1min

[STEP 4] SBERT batch encoding for 199147 pairs...


Batches:   0%|          | 0/195 [00:00<?, ?it/s]

Batches:   0%|          | 0/195 [00:00<?, ?it/s]

[INFO] Computing cosine similarities in chunks
Encoding done in 630.8s

[STEP 5] Filtering and clustering with DBSCAN
Pairs with detected plagiarism: 197

[STEP 6] Generating XML output...
[STEP 7] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00004 | GT: 0 | Det: 2 | P: 0.00 R: 1.00
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.92 R: 0.99
  Suspicious 00007 | GT: 6 | Det: 4 | P: 0.99 R: 0.49
  Suspicious 00008 | GT: 0 | Det: 2 | P: 0.00 R: 1.00
  Suspicious 00010 | GT: 6 | Det: 7 | P: 1.00 R: 0.29
  Suspicious 00011 | GT: 12 | Det: 19 | P: 0.96 R: 0.66
  Suspicious 00012 | GT: 14 | Det: 23 | P: 0.99 R: 0.55
  Suspicious 00014 | GT: 3 | Det: 3 | P: 0.63 R: 0.89
  Suspicious 00015 | GT: 11 | Det: 5 | P: 0.87 R: 0.41
  Suspicious 00016 | GT: 1 | Det: 3 | P: 0.32 R: 0.97
  Suspicious 00020 | GT: 3 | Det: 3 | P: 1.00 R: 0.44
  Suspicious 00024 | GT: 3 | Det: 4 | P: 0.46 R: 0.64
  Suspicious 00025 | GT: 1 | Det: 1 | P: 1.00 R: 0.15
  Susp

In [ ]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

for cell in nb.cells:
    if 'metadata' in cell:
        if 'executionInfo' in cell.metadata:
            del cell.metadata['executionInfo']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

Done!


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://55bbf0b3fb1d1c1df3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[SOURCES] Doc 00012 | GT sources: 1 | Target total: 30
[SOURCES] GT: [7150] | Random fill: 29 | Total: 30
[INIT] Loading paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Converting model to FP16 (Mixed Precision) for T4 Tensor Cores...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://55bbf0b3fb1d1c1df3.gradio.live
